In [1]:
# =============================================
# Developer Recommendation System
# Model: CatBoost
# Rule: If developer on leave → Not Selected
# =============================================

# 1️⃣ Upload Dataset

from google.colab import files
uploaded = files.upload()


# 2️⃣ Install Libraries

!pip install -q catboost
!pip install -q gradio


# 3️⃣ Imports

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import OrdinalEncoder

from catboost import CatBoostClassifier

import gradio as gr


# 4️⃣ Load Dataset

df = pd.read_excel("/content/devn dataset.xlsx")

print("Dataset Shape:",df.shape)

print("\nTarget Distribution:")
print(df["is_recommended"].value_counts(normalize=True))


# 5️⃣ Apply Hard Rule in Training

df.loc[df["dev_on_leave"]==True,"is_recommended"]=0


# 6️⃣ Split Features & Target

X=df.drop("is_recommended",axis=1).copy()

y=df["is_recommended"].copy()


# 7️⃣ Seniority Encoding

seniority_order=['junior','mid','senior']

ord_enc=OrdinalEncoder(
    categories=[seniority_order,seniority_order]
)

X[['required_seniority','dev_seniority']]=ord_enc.fit_transform(

    X[['required_seniority','dev_seniority']]

)


# 8️⃣ Categorical Columns

cat_features=[

'project_type',
'dev_specialty',
'dev_workload',
'dev_on_leave'

]

for col in cat_features:

    X[col]=X[col].astype(str)



# 9️⃣ Train Test Split

X_train,X_test,y_train,y_test=train_test_split(

X,
y,

test_size=0.20,

random_state=42,

stratify=y

)


print("\nTrain Size:",X_train.shape[0])
print("Test Size:",X_test.shape[0])



# 🔟 Train CatBoost Model

model=CatBoostClassifier(

iterations=600,

depth=6,

learning_rate=0.04,

cat_features=cat_features,

auto_class_weights='Balanced',

verbose=100,

random_seed=42,

early_stopping_rounds=50

)


model.fit(

X_train,

y_train,

eval_set=(X_test,y_test)

)



# 11️⃣ Evaluate Model

y_pred=model.predict(X_test)


print("\nAccuracy:",

round(accuracy_score(y_test,y_pred),4)

)


print("\nClassification Report\n")

print(classification_report(y_test,y_pred))



# 12️⃣ Feature Importance

importance=pd.Series(

model.get_feature_importance(),

index=X.columns

).sort_values(ascending=False)


print("\nFeature Importance\n")

print(importance)



# 13️⃣ Prediction Function

def predict_new(sample):


    # HARD RULE

    if sample["dev_on_leave"]=="Yes":

        return "❌ Not Selected (Developer On Leave)"


    df_new=pd.DataFrame([sample])


    # Convert Yes/No → Boolean

    df_new["dev_on_leave"]=df_new["dev_on_leave"].map(

        {

        "Yes":True,

        "No":False

        }

    )


    # Apply Encoding

    df_new[['required_seniority','dev_seniority']]=ord_enc.transform(

    df_new[['required_seniority','dev_seniority']]

    )


    for col in cat_features:

        df_new[col]=df_new[col].astype(str)



    pred=model.predict(df_new)[0]


    proba=model.predict_proba(df_new)[0][1]



    if pred==1:

        return f"✅ Selected (Confidence {proba:.1%})"


    else:

        return f"❌ Not Selected (Confidence {1-proba:.1%})"



# 14️⃣ Test Examples

examples=[


{"project_type":"web",

"required_seniority":"mid",

"dev_specialty":"web",

"dev_seniority":"senior",

"dev_workload":"light",

"dev_on_leave":"No",

"dev_tasks_this_week":1},


{"project_type":"game",

"required_seniority":"mid",

"dev_specialty":"game",

"dev_seniority":"senior",

"dev_workload":"light",

"dev_on_leave":"Yes",

"dev_tasks_this_week":0}

]


print("\nExample Predictions\n")


for e in examples:

    print(predict_new(e))



# 15️⃣ Gradio Interface


def gradio_predict(

project_type,

required_seniority,

dev_specialty,

dev_seniority,

dev_workload,

on_leave,

tasks

):


    sample={

"project_type":project_type,

"required_seniority":required_seniority,

"dev_specialty":dev_specialty,

"dev_seniority":dev_seniority,

"dev_workload":dev_workload,

"dev_on_leave":on_leave,

"dev_tasks_this_week":int(tasks)

    }


    return predict_new(sample)



interface=gr.Interface(

fn=gradio_predict,


inputs=[


gr.Dropdown(

["app","web","game"],

label="Project Type"

),


gr.Dropdown(

["junior","mid","senior"],

label="Required Seniority"

),


gr.Dropdown(

["app","web","game"],

label="Developer Specialty"

),


gr.Dropdown(

["junior","mid","senior"],

label="Developer Seniority"

),


gr.Dropdown(

["free","light","heavy"],

label="Workload"

),


gr.Dropdown(

["No","Yes"],

label="Developer On Leave"

),


gr.Number(

label="Tasks This Week",

value=1

)

],


outputs="text",


title="Developer Selection System",


description="AI decides if developer should be selected"

)


interface.launch()

Saving devn dataset.xlsx to devn dataset.xlsx
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.3 MB/s eta 0:00:00
Dataset Shape: (147, 8)

Target Distribution:
is_recommended
1    0.537415
0    0.462585
Name: proportion, dtype: float64

Train Size: 117
Test Size: 30
0:	learn: 0.6700741	test: 0.6676006	best: 0.6676006 (0)	total: 54.3ms	remaining: 32.5s
100:	learn: 0.0906557	test: 0.1161522	best: 0.1161522 (100)	total: 581ms	remaining: 2.87s
200:	learn: 0.0294957	test: 0.0438745	best: 0.0438745 (200)	total: 1.19s	remaining: 2.37s
300:	learn: 0.0148047	test: 0.0239306	best: 0.0239306 (300)	total: 1.79s	remaining: 1.78s
400:	learn: 0.0101944	test: 0.0170969	best: 0.0170969 (400)	total: 2.21s	remaining: 1.1s
500:	learn: 0.0077230	test: 0.0135538	best: 0.0135538 (500)	total: 2.44s	remaining: 482ms
599:	learn: 0.0062950	test: 0.0113619	best: 0.0113619 (599)	total: 2.67s	remaining: 0us

bestTest = 0.01136189412
bestIteration = 599


Accuracy: 1.0

Classification Report

             